In [119]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

In [120]:
# --- Clasificar edades ---
def clasificar_edad(anio_nac):
    if pd.isna(anio_nac):
        return "Desconocido"
    try:
        anio_nac = int(anio_nac)
    except:
        return "Desconocido"
    if anio_nac >= 1995:
        return "Joven"
    elif 1975 <= anio_nac <= 1994:
        return "Adulto"
    else:
        return "Mayor"

# --- Muestreo proporcional ---
def muestrear_clientes(df, n_muestra, random_state=42):
    import warnings
    warnings.filterwarnings("ignore", category=DeprecationWarning)

    df = df.copy()
    df["RangoEdad"] = df["Fecha_Nacimiento"].apply(clasificar_edad)
    total = len(df)

    if total <= n_muestra:
        print(f"Total {total} <= {n_muestra}: se devuelven todos los clientes.")
        return df

    df_resultado = (
        df.groupby(["CLISexo", "UbicacionPreferida", "RangoEdad"], group_keys=False)
          .apply(lambda x: x.sample(
              n=min(len(x), max(1, round(len(x)/total * n_muestra))),
              random_state=random_state
          ))
    )

    # Ajustar por exceso o defecto
    if len(df_resultado) > n_muestra:
        df_resultado = df_resultado.sample(n=n_muestra, random_state=random_state)
    elif len(df_resultado) < n_muestra:
        faltan = n_muestra - len(df_resultado)
        extra = df.sample(n=faltan, random_state=random_state)
        df_resultado = pd.concat([df_resultado, extra])

    print(f"Muestreo completado: {len(df_resultado)} clientes (objetivo {n_muestra}).")
    return df_resultado


# --- Guardar siempre en el mismo archivo/hoja ---
def save_to_excel(df, filename="clientes.xlsx", sheet_name="clientes"):
    with pd.ExcelWriter(filename, engine="openpyxl", mode="w") as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"Archivo guardado: {filename} (hoja {sheet_name})")


In [121]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Comerssia;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

query = """
--CLIENTES DIAMANTES MUY ACTIVOS (GRUPO 1)
SELECT DISTINCT s.Cliente,
	cc.Documento,
	cc.Nombres,
	cc.Apellidos,
	cc.Email,
	cc.Celular,
	cp.UbicacionPreferida, 
	YEAR(CLIFechaNacimiento) AS Fecha_Nacimiento,
	CLISexo ,s.Segmento + ' ' + s.Recencia AS Segmento
FROM Ventas_Comerssia.dbo.Segmentacion_Clientes s
INNER JOIN Ventas_Comerssia.dbo.Cliente_Perfil cp ON s.Cliente = cp.Cliente
INNER JOIN Ventas_Comerssia.dbo.Clientes_Comerssia c ON s.Cliente = c.ClienteLimpio
INNER JOIN Ventas_Comerssia.dbo.View_Contacto_Clientes cc ON c.ClienteLimpio = cc.Cliente
WHERE Segmento = 'Diamante'
	AND (CLIHabeas = 1)
	AND (Recencia = 'Activo' OR Recencia = 'Muy Activo')
	AND CLIFechaNacimiento IS NOT NULL
	AND CLISexo IS NOT NULL
	AND UbicacionPreferida IN ('LOCCITANE ANDINO','LOCCITANE EL DORADO','LOCCITANE GRAN ESTACION',
		'LOCCITANE PARQUE LA COLINA','LOCCITANE SANTA ANA','LOCCITANE UNICENTRO','LOCCITANE UNICENTRO P1')
	AND Contacto LIKE '%Cel%'
	AND TipoIdentificacion != 'NI'
"""
# Ejecutar y cargar en DataFrame
df = pd.read_sql(query, engine)

print("Total clientes consulta:", len(df))
df.head()

n_muestra = 2000
resultado = muestrear_clientes(df, n_muestra)
save_to_excel(resultado) 

Total clientes consulta: 1307
Total 1307 <= 2000: se devuelven todos los clientes.
Archivo guardado: clientes.xlsx (hoja clientes)


In [122]:
# # ---------------------------------
# # 1. Configuración
# # ---------------------------------

# n_muestra = 700 # Número objetivo de clientes
# nombre_archivo = "clientes.xlsx"  # Archivo de salida

# params = urllib.parse.quote_plus(
#     "DRIVER={ODBC Driver 17 for SQL Server};"
#     "SERVER=181.57.189.150,1433;"
#     "DATABASE=Ventas_Comerssia;"
#     "UID=sa;"
#     "PWD=P@ssw0rd*;"
# )

# # Crear el motor de conexión
# engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# query = """
# SELECT DISTINCT s.Cliente, cp.UbicacionPreferida, YEAR(CLIFechaNacimiento) AS Fecha_Nacimiento, CLISexo ,s.Segmento + ' ' + s.Recencia AS Segmento
# FROM Ventas_Comerssia.dbo.Segmentacion_Clientes s
# INNER JOIN Ventas_Comerssia.dbo.Cliente_Perfil cp ON s.Cliente = cp.Cliente
# INNER JOIN Ventas_Comerssia.dbo.Clientes_Comerssia c ON s.Cliente = c.ClienteLimpio
# INNER JOIN Ventas_Comerssia.dbo.View_Contacto_Clientes cc ON c.ClienteLimpio = cc.Cliente
# WHERE Segmento IN ('Diamante','Oro','Plata')
# 	AND Recencia = 'Churn'
# 	AND UbicacionPreferida IN ('LOCCITANE ANDINO','LOCCITANE EL DORADO','LOCCITANE GRAN ESTACION',
# 		'LOCCITANE PARQUE LA COLINA','LOCCITANE SANTA ANA','LOCCITANE UNICENTRO','LOCCITANE UNICENTRO P1')
# 	AND Contacto LIKE '%Cel%'
# """
# # Ejecutar y cargar en DataFrame
# df = pd.read_sql(query, engine)

# df.head()

In [123]:
# def clasificar_edad(anio_nac):
#     """Clasifica año de nacimiento en Joven/Adulto/Mayor/Desconocido."""
#     if pd.isna(anio_nac):
#         return "Desconocido"
#     try:
#         anio_nac = int(anio_nac)
#     except:
#         return "Desconocido"
    
#     if anio_nac >= 1995:
#         return "Joven"
#     elif 1975 <= anio_nac <= 1994:
#         return "Adulto"
#     else:
#         return "Mayor"


In [124]:
# # ---------------------------
# # 3. Muestreo proporcional
# # ---------------------------
# n_muestra = 700

# df_balanceado = (
#     df.groupby(["CLISexo","UbicacionPreferida","RangoEdad"], group_keys=False, as_index=False)
#       .apply(lambda x: x.sample(
#           n=max(1, round(len(x) / len(df) * n_muestra)),
#           random_state=42
#       ))
# )

# # Ajuste por redondeo (por si sobra o falta alguno)
# if len(df_balanceado) > n_muestra:
#     df_balanceado = df_balanceado.sample(n_muestra, random_state=42)
# elif len(df_balanceado) < n_muestra:
#     faltan = n_muestra - len(df_balanceado)
#     extra = df.sample(n=faltan, random_state=42)
#     df_balanceado = pd.concat([df_balanceado, extra])


In [125]:
# # ---------------------------
# # 4. Exportar resultado
# # ---------------------------
# df_balanceado.to_excel("clientes_balanceados.xlsx", index=False)

# print("✅ Muestreo completo. Total clientes seleccionados:", len(df_balanceado))